# Vectors and Vector Spaces

| Difficulty | ██░░░░░░░░ 2/10 |
| :--- | :--- |
| Prerequisites | 00.01–00.04 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/01_vectors_and_spaces.ipynb)

---

## 🎯 Learning Objective

Understand vectors as objects that live in vector spaces, and learn the norms and similarity measures used to compare them.

---

## 📐 Theory

A **vector** has two equivalent faces: algebraically, it's a list of numbers
$\mathbf{x} = (x_1, \dots, x_n)$; geometrically, it's an arrow from the origin to a point in
space. Both views matter — the algebra is what you compute with, the geometry is what
you should *picture* while computing.

### The dot product

The dot product combines two vectors into a single number:

$$\mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^{n} u_i v_i = \|\mathbf{u}\|\,\|\mathbf{v}\|\cos\theta$$

The second equality is the geometric payoff: the dot product encodes *both* how long the
vectors are *and* the angle $\theta$ between them.

### Norms: measuring "size"

A norm turns a vector into a single non-negative number representing its magnitude. Three
show up constantly in ML:

$$\|\mathbf{x}\|_1 = \sum_i |x_i| \qquad
\|\mathbf{x}\|_2 = \sqrt{\sum_i x_i^2} \qquad
\|\mathbf{x}\|_\infty = \max_i |x_i|$$

$L^1$ ("Manhattan") norm sums absolute values — it's robust to outliers and encourages
*sparsity* when used as a regularizer. $L^2$ ("Euclidean") norm is straight-line distance —
the default notion of length. $L^\infty$ only cares about the single largest entry.

### Cosine similarity

Dividing out the magnitudes from the dot product isolates pure *direction*:

$$\cos\theta = \frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|}$$

This ranges from $-1$ (opposite directions) to $1$ (same direction), completely ignoring how
long each vector is — which is exactly why it's the standard similarity measure for embeddings
that can vary wildly in magnitude.

### Vector spaces and subspaces

A **vector space** is a set closed under vector addition and scalar multiplication — combine
any two members (or scale one) and you stay inside the set. A **subspace** is a subset of a
vector space that is itself a vector space (in particular, it must contain the origin). The
**span** of a set of vectors is the subspace of all their possible linear combinations, and a
set of vectors is **linearly independent** if none of them can be written as a combination of
the others — each one adds a genuinely new direction.

---

## 👁️ Visual Intuition

Plotting two vectors as arrows from the origin makes the dot product's geometric meaning
concrete: the norms are the arrow lengths, and $\cos\theta$ is read directly off the angle
between them.

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

u = np.array([3.0, 1.0])
v = np.array([1.0, 2.0])
cos_theta = (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))

fig, ax = plt.subplots(figsize=(5, 5))
ax.quiver(0, 0, u[0], u[1], angles='xy', scale_units='xy', scale=1, color="#4C72B0", width=0.012, label="u")
ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color="#DD8452", width=0.012, label="v")
ax.set_xlim(-1, 4); ax.set_ylim(-1, 4)
ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.legend()
ax.set_title(f"||u||={np.linalg.norm(u):.2f}   ||v||={np.linalg.norm(v):.2f}   cos(theta)={cos_theta:.2f}")
plt.show()

## 🐍 Implementation from Scratch

Every quantity from the theory section is a one-line loop. Implementing them by hand first,
then checking against NumPy, builds the muscle memory to read vectorized code fluently.

In [ ]:
# Dot product, norms, and cosine similarity -- pure Python, then cross-checked with NumPy
import numpy as np

def dot(u, v):
    return sum(ui * vi for ui, vi in zip(u, v))

def l1_norm(x):
    return sum(abs(xi) for xi in x)

def l2_norm(x):
    return sum(xi ** 2 for xi in x) ** 0.5

def linf_norm(x):
    return max(abs(xi) for xi in x)

def cosine_similarity(u, v):
    # Guard against a ZERO vector: l2_norm(u)=0 or l2_norm(v)=0 would otherwise raise
    # ZeroDivisionError (pure Python) here -- a real case in practice (e.g. an unused
    # padding/OOV embedding row that was never trained and stayed at all-zero init).
    # Cosine similarity is mathematically undefined for a zero vector (there's no angle
    # to measure), so we return exactly 0.0 -- "no meaningful similarity" -- rather than
    # letting the exception propagate or silently producing garbage.
    norm_u, norm_v = l2_norm(u), l2_norm(v)
    if norm_u == 0 or norm_v == 0:
        return 0.0
    return dot(u, v) / (norm_u * norm_v)

rng = np.random.default_rng(0)
a, b = rng.standard_normal(6), rng.standard_normal(6)

print("dot(a,b)          :", round(dot(a, b), 4), " vs np.dot:", round(float(a @ b), 4))
print("L1 norm of a       :", round(l1_norm(a), 4), " vs numpy:", round(float(np.linalg.norm(a, 1)), 4))
print("L2 norm of a       :", round(l2_norm(a), 4), " vs numpy:", round(float(np.linalg.norm(a, 2)), 4))
print("Linf norm of a     :", round(linf_norm(a), 4), " vs numpy:", round(float(np.linalg.norm(a, np.inf)), 4))
print("cosine_similarity  :", round(cosine_similarity(a, b), 4))

# --- Regression test: a zero vector (e.g. an untrained padding embedding) must not crash ---
zero_vec = [0.0] * 6
print("\ncosine_similarity(a, zero_vector) =", cosine_similarity(a, zero_vec), "(defined as 0.0, not a crash)")

## 🤖 Why This Matters for AI

**Word embeddings** map words to vectors such that geometric relationships mirror semantic
ones. **Cosine similarity** is the standard way to compare them — because it ignores
magnitude, two embeddings pointing the same direction are "similar" regardless of how
confidently-scaled either one happens to be. This is the exact computation behind semantic
search and retrieval-augmented generation: embed the query, embed every document, rank by
cosine similarity.

Below, we hand-craft toy embeddings where `queen = king - man + woman` *exactly*, to make the
famous word2vec-style analogy concrete and verifiable rather than mysterious.

In [ ]:
# Cosine similarity powering semantic search + the classic word-vector analogy
import numpy as np

# Toy 4D "embeddings" -- in a real model these are learned, but here we hand-craft them so the
# geometric relationships are exact and easy to verify by hand.
embeddings = {
    "king":   np.array([0.80, 0.90, 0.10, 0.00]),
    "man":    np.array([0.60, 0.10, 0.10, 0.00]),
    "woman":  np.array([0.60, 0.90, 0.10, 0.00]),
    "queen":  np.array([0.80, 1.70, 0.10, 0.00]),   # == king - man + woman, by construction
    "apple":  np.array([0.00, 0.00, 0.90, 0.70]),
    "orange": np.array([0.00, 0.10, 0.80, 0.90]),
}

def cosine_similarity(u, v):
    # Same zero-vector guard as the pure-Python version above: np.linalg.norm(u)=0 would
    # otherwise produce a 0/0 division (nan, with a RuntimeWarning) rather than a clear,
    # intentional "undefined similarity" result.
    norm_u, norm_v = np.linalg.norm(u), np.linalg.norm(v)
    if norm_u == 0 or norm_v == 0:
        return 0.0
    return (u @ v) / (norm_u * norm_v)

# --- Semantic search: rank every word by similarity to a "fruit-like" query vector ---
query = np.array([0.05, 0.05, 0.85, 0.75])
ranked = sorted(embeddings.items(), key=lambda kv: cosine_similarity(query, kv[1]), reverse=True)
print("Semantic search results (most similar to query first):")
for word, vec in ranked:
    print(f"  {word:8s} cosine similarity = {cosine_similarity(query, vec):.3f}")

# --- Analogy: king - man + woman =~ queen ---
analogy_vector = embeddings["king"] - embeddings["man"] + embeddings["woman"]
candidates = {w: cosine_similarity(analogy_vector, e) for w, e in embeddings.items() if w not in ("king", "man", "woman")}
best_word, best_sim = max(candidates.items(), key=lambda kv: kv[1])
print(f"\n'king - man + woman' is most similar to: '{best_word}'  (cosine similarity={best_sim:.3f})")

# --- Regression test: an untrained/padding embedding (all-zero vector) must not crash
# semantic search -- a realistic occurrence for an OOV token or an unused vocabulary slot. ---
padding_embedding = np.zeros(4)
sim_to_padding = cosine_similarity(query, padding_embedding)
print(f"\ncosine_similarity(query, zero/padding embedding) = {sim_to_padding}  (0.0, not nan)")
assert np.isfinite(sim_to_padding), "cosine_similarity must stay finite for a zero vector"

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $\mathbf{x} = (3, -4, 0, 1)$, compute $\|\mathbf{x}\|_1$, $\|\mathbf{x}\|_2$, and
   $\|\mathbf{x}\|_\infty$ by hand, then verify with `l1_norm`/`l2_norm`/`linf_norm`. As a sanity
   check, confirm the ordering $\|\mathbf{x}\|_\infty \le \|\mathbf{x}\|_2 \le \|\mathbf{x}\|_1$
   holds for your answer — this ordering holds for *every* vector, not just this one.

<details>
<summary>💡 Solution</summary>

$\|\mathbf{x}\|_1 = |3|+|-4|+|0|+|1| = 8$. $\|\mathbf{x}\|_2 = \sqrt{9+16+0+1}=\sqrt{26}\approx
5.099$. $\|\mathbf{x}\|_\infty = \max(3,4,0,1)=4$. All three match `l1_norm(x)`, `l2_norm(x)`,
`linf_norm(x)` exactly. Ordering check: $4 \le 5.099 \le 8$ ✓ — and this holds in general, not
just here: the single largest entry can't exceed the vector's $L^2$ length (it's only one of the
terms under the square root), and squaring the $L^1$ sum picks up strictly nonnegative cross
terms beyond what $L^2$ squared has ($(\sum_i|x_i|)^2 = \sum_i x_i^2 + \sum_{i\ne j}|x_i||x_j|
\ge \sum_i x_i^2$ — concretely here, $8^2=64$ versus $26$, a cross-term contribution of $38$),
so $\|\mathbf{x}\|_1\ge\|\mathbf{x}\|_2$ always too.

</details>

### 💭 UNDERSTAND
2. A query vector $q=(1,0)$ is compared against two candidates, $a=(3,3)$ and $b=(1,0.1)$.
   Without computing anything yet, which candidate do you expect to have the larger *dot
   product* with $q$, and which the larger *cosine similarity*? Compute both to check, then
   explain why the two rankings disagree.

<details>
<summary>💡 Solution</summary>

$\mathbf{q}\cdot\mathbf{a}=3$ and $\mathbf{q}\cdot\mathbf{b}=1$ — dot product ranks $a$ higher.
But $\cos(q,a)=\frac{3}{1\cdot\sqrt{18}}\approx0.707$ ($45°$ from $q$) while
$\cos(q,b)=\frac{1}{1\cdot\sqrt{1.01}}\approx0.995$ (about $5.7°$ from $q$) — cosine similarity
ranks $b$ higher, decisively. The rankings disagree because dot product conflates *alignment*
with *length*: $a$ wins on dot product largely because $\|a\|\approx4.24$ is much longer than
$\|b\|\approx1.00$, even though $b$ points almost exactly where $q$ points and $a$ is $45°$ off.
Cosine similarity divides the length back out, so it measures direction alone — exactly the
property the Theory section relies on when comparing embeddings of words with very different
training frequencies (and hence very different norms).

</details>

### ✏️ DERIVE
3. The Theory section states cosine similarity "ranges from $-1$ ... to $1$" without proving it.
   Derive this bound: for nonzero $\mathbf{u},\mathbf{v}\in\mathbb{R}^n$, prove the
   Cauchy-Schwarz inequality $|\mathbf{u}\cdot\mathbf{v}| \le \|\mathbf{u}\|_2\|\mathbf{v}\|_2$
   (equivalently, $|\cos\theta|\le1$) using the fact that $f(t)=\|\mathbf{u}-t\mathbf{v}\|_2^2\ge0$
   for every real $t$.

<details>
<summary>💡 Solution outline</summary>

Expand $f(t) = (\mathbf{u}-t\mathbf{v})\cdot(\mathbf{u}-t\mathbf{v}) =
\|\mathbf{v}\|^2t^2 - 2(\mathbf{u}\cdot\mathbf{v})t + \|\mathbf{u}\|^2$ — a quadratic in $t$ with
leading coefficient $\|\mathbf{v}\|^2\ge0$. Since $f(t)\ge0$ for *every* real $t$ (it's a squared
norm), this quadratic cannot have two distinct real roots, so its discriminant must satisfy
$$4(\mathbf{u}\cdot\mathbf{v})^2 - 4\|\mathbf{u}\|^2\|\mathbf{v}\|^2 \le 0
\;\;\Longrightarrow\;\; (\mathbf{u}\cdot\mathbf{v})^2\le\|\mathbf{u}\|^2\|\mathbf{v}\|^2
\;\;\Longrightarrow\;\; |\mathbf{u}\cdot\mathbf{v}|\le\|\mathbf{u}\|\|\mathbf{v}\|.$$
Dividing both sides by $\|\mathbf{u}\|\|\mathbf{v}\|>0$ gives
$|\cos\theta|=\left|\frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\|\mathbf{v}\|}\right|\le1$
exactly as claimed. Numeric check with $u=(1,2,-1)$, $v=(2,0,1)$: $u\cdot v=1$, $\|u\|^2=6$,
$\|v\|^2=5$, so $\cos\theta=1/\sqrt{30}\approx0.1826$, comfortably inside $[-1,1]$. Scaling
$u\to10u$ leaves $\cos\theta$ exactly unchanged (numerator and denominator both pick up the same
factor of $10$ and it cancels), while $u\to-10u$ flips its sign to $\approx-0.1826$ — confirming
cosine similarity depends only on each vector's *direction*, never on its length or a positive
rescaling of it.

</details>

### 🐍 IMPLEMENT
4. Write two new functions, `most_similar_pair(vectors)` and `most_dissimilar_pair(vectors)`,
   that take a list of vectors and return the index pair achieving the highest and lowest
   `cosine_similarity`, respectively. Test them on 5 random vectors in $\mathbb{R}^4$ (e.g.
   `rng.standard_normal(4)`, called 5 times).

<details>
<summary>✅ How to know you're right</summary>

Your functions' output must agree exactly with a brute-force double loop over every pair
(`itertools.combinations(range(5), 2)`) — there's no shortcut here that a full pairwise scan
doesn't also verify, so any disagreement means a bug in yours. As a normalization sanity check,
confirm `cosine_similarity(v, v)` returns exactly `1.0` (up to floating-point rounding) for every
one of your 5 vectors — self-similarity must always be maximal — and confirm every pairwise
similarity you compute lies in $[-1, 1]$: exactly the Cauchy-Schwarz bound from exercise 3, now
checked empirically instead of proven.

</details>

### 🤖 APPLY
5. A recommendation system embeds users and items in a shared space. Consider a user vector and
   two candidate items: a "niche" item with a *small* embedding norm (few training interactions)
   whose direction is nearly identical to the user's, versus a "popular" item with a *large*
   embedding norm (many training interactions) whose direction is only partially aligned with
   the user's. Compute both the dot product and the cosine similarity between the user and each
   item, and compare the two rankings.

<details>
<summary>✅ What you should observe</summary>

With e.g. `user=(1, 0.5, 0.2, 0)`, a niche item at roughly `(0.37, 0.16, 0.06, 0.03)` (small
norm, ≈0.995 cosine similarity to the user — nearly perfectly aligned), and a popular item at
roughly `(3.0, 0.6, 5.1, 2.4)` (large norm, ≈0.593 cosine similarity — only partially aligned):
the dot product ranks the *popular* item higher (≈4.32 vs ≈0.46) simply because its larger norm
inflates the raw product, while cosine similarity ranks the *niche* item higher, decisively,
because it ignores that norm difference entirely and sees only direction. This is exactly the
popularity bias real recommender systems have to actively correct for: raw dot-product-based
retrieval (common in large-scale approximate nearest-neighbor search, chosen for speed) can
systematically favor popular items whose embeddings simply grew larger from more training
updates, not because they are a genuinely better match for this particular user.

</details>

### 🚀 CHALLENGE
6. Extend the `embeddings` dictionary with 3 new words of your own that satisfy an analogy of
   your choosing (e.g. a country/capital relationship), and verify with cosine similarity that
   the analogy vector's nearest neighbor is the word you intended. Then investigate robustness:
   how much random noise can you add to your target word's embedding before some *other* word
   in the dictionary becomes the nearest neighbor instead?

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer constructs an analogy with a clear geometric structure (e.g.
$\text{capital}_1 - \text{country}_1 + \text{country}_2 \approx \text{capital}_2$), mirroring the
notebook's own `king - man + woman ≈ queen` construction, and confirms it numerically. For that
original construction, the margin between `queen`'s similarity to the analogy vector (exactly
$1.0$, since `queen` was built to equal it exactly) and the runner-up (`orange`, at
$\approx0.110$) is large — about $0.89$; a strong answer's own construction should show a
comparably clear margin, not a coincidental near-tie. Investigating robustness (e.g. adding
`rng.standard_normal(4) * noise_scale` to the target word's embedding and re-checking the ranking
over many trials) reveals the analogy is robust to small perturbations but degrades gradually as
noise grows: for the original king/man/woman/queen construction, perturbations up to roughly
`noise_scale≈0.3–0.5` essentially never break the ranking, while by `noise_scale≈1.5–2.0` the
intended word loses to a competitor in roughly 10–25% of trials. A strong answer connects this to
a practical point: real learned embeddings are never mathematically exact the way this toy
example is, so an analogy's reliability in practice depends on exactly this kind of margin — how
much room the intended answer has over its nearest competitors — not on the relationship holding
exactly.

</details>

---

## 📚 Further Reading
- Mikolov et al., ["Efficient Estimation of Word Representations in Vector Space"](https://arxiv.org/abs/1301.3781) (word2vec)
- Deisenroth, Faisal & Ong, *Mathematics for Machine Learning*, Ch. 3 (Analytic Geometry)

---

*Next notebook: [Matrix Operations](02_matrix_operations.ipynb)*